# v0.34.0 — Model table DDL

## 1. Intro

Since v0.34.0 a model declares its table's schema mode, change feed, permissions and comment with the full
SurrealDB-ORM's keys, and `define_table()` writes its `DEFINE TABLE` and one `DEFINE FIELD` per column and applies
them:

- `SurrealConfigDict(schema_mode=…, changefeed=…, permissions=…, comment=…, flexible_fields=…)` declares the table's
  clauses, checked when the class is defined. `SchemaMode` is the full ORM's enum, and v0.33.0's `table_type` takes
  effect: `TYPE NORMAL` or `TYPE ANY`, and `TableType.USER` makes the table `SCHEMAFULL`;
- each column's `TYPE` comes from its annotation: `option<…>` for an optional field, a constant default as `DEFAULT`,
  one column per field of a sub-model, `FLEXIBLE` on a free-form object of a `SCHEMAFULL` table, an item definition
  for an untyped list;
- `table_ddl()` renders the statements without a connection; `await define_table()` applies them and returns them,
  keeping the records, fields, clauses, indexes and events the model does not declare;
- `get_schema_mode()`, `get_changefeed()` and `get_permissions()` read the declaration back.

The code is the same on SurrealDB 2.x, 3.x and the embedded engine. Only the rows of the README's 2.x/3.x table differ:
a write naming a field a `SCHEMAFULL` table does not define is refused on 3.x and dropped on 2.x (§6), and on a
`SCHEMAFULL` table 2.x needs `FLEXIBLE` on the columns that hold `any` and on a list of sub-models holding a `FLEXIBLE`
column, which `define_table()` adds there (§8). The notebook runs top to bottom on both lines.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The tables live in
their own database, `examples_table_ddl`, reached through the named connection `tdl_demo`. The notebook starts by
resetting that database, so it can be re-executed in place: `REMOVE DATABASE` and `DEFINE DATABASE` go in **one**
query, because on SurrealDB 3.x a bare `REMOVE DATABASE` leaves the session failing until the database is defined
again.

`apply_ddl()` runs SurrealQL through the SDK client's `query_raw()` and checks the status of **every** statement
(`client.query()` checks only the first one's); `table_definition()` reads a table's definition back from
`INFO FOR DB`. `major` is the server's major version: §6 and §8 print what each line does.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # the database this notebook defines its tables in
    "tdl_demo",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_table_ddl",
)


async def apply_ddl(client, ddl: str) -> int:
    """Run SurrealQL statements and check every one of them; return how many ran."""
    response = await client.query_raw(ddl, {})
    if "error" in response:
        raise RuntimeError(response["error"])
    refused = [result["result"] for result in response["result"] if result["status"] != "OK"]
    if refused:
        raise RuntimeError(refused)
    return len(response["result"])


async def table_definition(table: str) -> str:
    """The definition of *table*, as INFO FOR DB reads it back."""
    return (await demo.query("INFO FOR DB;", {}))["tables"][table]


demo = await SurrealDBConnectionManager.get_client("tdl_demo")
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_table_ddl; DEFINE DATABASE examples_table_ddl;")
major = int((await demo.version()).rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print("Connections:", SurrealDBConnectionManager.list_connections())

Connections: ['default', 'tdl_demo']


## 3. Models

Every model runs on the connection `tdl_demo`:

- `Person` is the README's model (section 35): a `SCHEMAFULL` table with a change feed, permissions and a comment.
  `permissions` maps an operation to `"FULL"`, `"NONE"`, `"WHERE <condition>"` or a bare condition, the full ORM's
  form; an operation left out is `NONE`, SurrealDB's default.
- `Note` declares none of the new keys: `define_table()` keeps what the database has (§7).
- `Log` holds records and edges (`TableType.ANY`) and any field (`SchemaMode.SCHEMALESS`).
- `Likes` is an edge model (v0.22.0, v0.33.0) whose `SCHEMAFULL` table has a data field, `stars`.
- `NoteStats` is a materialized view (v0.32.0): it takes `permissions`, `changefeed` and `comment`.

The keys are checked when the class is defined: a value of the wrong type raises `TypeError` — a plain-string
`schema_mode`, even `"SCHEMAFULL"`, included — and a value of the wrong form `ValueError`.

In [2]:
from datetime import datetime
from enum import StrEnum
from typing import Any

from pydantic import BaseModel, Field
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealEdge, BaseSurrealModel, SchemaMode, SurrealConfigDict, TableType


class Tier(StrEnum):
    FREE = "free"
    PRO = "pro"


class Address(BaseModel):
    city: str
    zip: str | None = None


class Person(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="person",
        connection="tdl_demo",
        schema_mode=SchemaMode.SCHEMAFULL,
        changefeed="7d",
        permissions={"select": "FULL", "create": "$auth.id != NONE"},
        comment="People",
    )

    id: str | None = None
    name: str
    age: int = 18
    nickname: str | None = None
    tier: Tier = Tier.FREE
    tags: list[str] = Field(default_factory=list)
    address: Address | None = None
    meta: dict[str, Any] = Field(default_factory=dict)
    password: str = Field(alias="password_hash")


class Note(BaseSurrealModel):  # none of the new keys
    model_config = SurrealConfigDict(table_name="note", connection="tdl_demo", server_fields=["created"])

    id: str | None = None
    body: str
    email: str | None = None
    created: datetime | None = None
    done: bool = False


class Log(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="log", connection="tdl_demo", table_type=TableType.ANY, schema_mode=SchemaMode.SCHEMALESS
    )

    id: str | None = None
    payload: dict[str, Any] = Field(default_factory=dict)
    detail: Any = None


class Likes(BaseSurrealEdge):
    model_config = SurrealConfigDict(
        table_name="likes",
        connection="tdl_demo",
        relation_in="person",
        relation_out="note",
        schema_mode=SchemaMode.SCHEMAFULL,
        permissions={"select": "FULL"},
    )

    stars: int = 1


class NoteStats(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="note_stats",
        connection="tdl_demo",
        view_query="SELECT count() AS n FROM note GROUP ALL",
        comment="Counts",
        permissions={"select": "FULL"},
    )

    id: RecordID | None = None
    n: int


try:

    class Strict(BaseSurrealModel):
        model_config = SurrealConfigDict(table_name="strict", schema_mode="SCHEMAFULL")  # a plain string

except TypeError as error:
    print(error)

try:

    class Reader(BaseSurrealModel):
        model_config = SurrealConfigDict(table_name="reader", permissions={"read": "FULL"})  # not an operation

except ValueError as error:
    print(error)

Strict: model_config['schema_mode'] must be a surreal_orm_lite.SchemaMode, got str.
Reader: model_config['permissions'] names the operation 'read'; a table's operations are select, create, update and delete.


## 4. Render and read the declaration

`table_ddl()` renders the statements without a connection, one per item: the table statement, then one
`DEFINE FIELD` per column, parents before children. The table's clauses are what the model declares. Each column's
`TYPE` comes from its annotation: `option<…>` for `nickname: str | None`, the enum's values for `tier`, one column per
field of `Address` after `address`, `FLEXIBLE` on the free-form `meta` object (it keeps the keys the model does not
declare), the alias `password_hash` as the column. A constant default of a column that is not optional is its
`DEFAULT`.

The getters read the declaration back: `get_permissions()` returns the declared mapping as given, a copy; `{}` when
the model declares none. `get_schema_mode()` is `None` when the mode is not declared: `define_table()` then keeps the
stored one.

In [3]:
print(*Person.table_ddl(), sep="\n")
print()
print("schema mode:", repr(Person.get_schema_mode()))
print("change feed:", Person.get_changefeed())
print("permissions:", Person.get_permissions())
print("Note:", Note.get_schema_mode(), Note.get_changefeed(), Note.get_permissions())
print("Log:", repr(Log.get_schema_mode()), "| table type:", repr(Log.get_table_type()))

DEFINE TABLE OVERWRITE `person` TYPE NORMAL SCHEMAFULL COMMENT 'People' CHANGEFEED 7d PERMISSIONS FOR select FULL, FOR create WHERE $auth.id != NONE, FOR update, delete NONE;
DEFINE FIELD OVERWRITE `name` ON `person` TYPE string;
DEFINE FIELD OVERWRITE `age` ON `person` TYPE int DEFAULT 18;
DEFINE FIELD OVERWRITE `nickname` ON `person` TYPE option<string>;
DEFINE FIELD OVERWRITE `tier` ON `person` TYPE 'free' | 'pro' DEFAULT 'free';
DEFINE FIELD OVERWRITE `tags` ON `person` TYPE array<string>;
DEFINE FIELD OVERWRITE `address` ON `person` TYPE option<object>;
DEFINE FIELD OVERWRITE `address`.`city` ON `person` TYPE string;
DEFINE FIELD OVERWRITE `address`.`zip` ON `person` TYPE option<string>;
DEFINE FIELD OVERWRITE `meta` ON `person` TYPE object FLEXIBLE;
DEFINE FIELD OVERWRITE `password_hash` ON `person` TYPE string;

schema mode: <SchemaMode.SCHEMAFULL: 'SCHEMAFULL'>
change feed: 7d
permissions: {'select': 'FULL', 'create': '$auth.id != NONE'}
Note: None None {}
Log: <SchemaMode.SCHE

## 5. Apply it

`define_table()` reads the table's definition (`INFO FOR DB`, and `INFO FOR TABLE` when it exists), then sends one
statement per call: the `DEFINE TABLE`, then the fields. It returns the statements applied, table first, joined with
`"\n"` — the full ORM's return. On a new table they are `table_ddl()`'s. The call is idempotent: run it at start-up.

v0.30.0's `DatabaseIntrospector` reads the table back. SurrealDB renders the change feed's duration in its own units:
`7d` reads back `1w`.

In [4]:
from surreal_orm_lite import DatabaseIntrospector

sql = await Person.define_table()
print(sql)
print()
print("the statements of table_ddl():", sql == "\n".join(Person.table_ddl()))
print("a second call sends the same:", await Person.define_table() == sql)
print()

schema = await DatabaseIntrospector("tdl_demo", tables=["person"]).introspect()
state = schema.tables["person"]
print("schema mode:", state.schema_mode, "| change feed:", state.changefeed, "| comment:", state.comment)
print("permissions:", dict(state.permissions))
for name in ("age", "nickname", "tier", "address.city", "meta"):
    field = state.fields[name]
    print(f"  {name}: {field.field_type}, nullable={field.nullable}, default={field.default}, flexible={field.flexible}")

DEFINE TABLE OVERWRITE `person` TYPE NORMAL SCHEMAFULL COMMENT 'People' CHANGEFEED 7d PERMISSIONS FOR select FULL, FOR create WHERE $auth.id != NONE, FOR update, delete NONE;
DEFINE FIELD OVERWRITE `name` ON `person` TYPE string;
DEFINE FIELD OVERWRITE `age` ON `person` TYPE int DEFAULT 18;
DEFINE FIELD OVERWRITE `nickname` ON `person` TYPE option<string>;
DEFINE FIELD OVERWRITE `tier` ON `person` TYPE 'free' | 'pro' DEFAULT 'free';
DEFINE FIELD OVERWRITE `tags` ON `person` TYPE array<string>;
DEFINE FIELD OVERWRITE `address` ON `person` TYPE option<object>;
DEFINE FIELD OVERWRITE `address`.`city` ON `person` TYPE string;
DEFINE FIELD OVERWRITE `address`.`zip` ON `person` TYPE option<string>;
DEFINE FIELD OVERWRITE `meta` ON `person` TYPE object FLEXIBLE;
DEFINE FIELD OVERWRITE `password_hash` ON `person` TYPE string;

the statements of table_ddl(): True
a second call sends the same: True

schema mode: SCHEMAFULL | change feed: 1w | comment: People
permissions: {'create': 'WHERE $auth.

## 6. Writes

The ORM writes what the model declares, so a record round-trips on a `SCHEMAFULL` table: the defaults are applied,
the sub-model's fields kept, and `meta`, `FLEXIBLE`, keeps its nested keys.

In [5]:
ada = Person(id="ada", name="Ada", address=Address(city="Paris"), meta={"plan": {"seats": 3}}, password="hash")
await ada.save()
stored = await Person.objects().get("ada")
print("read back equal:", stored.model_dump() == ada.model_dump())
print(stored.age, stored.tier, stored.address.city, stored.meta)

read back equal: True
18 free Paris {'plan': {'seats': 3}}


A write naming a field the table does not define — `merge()` with a key the model does not declare, a raw query — is
**refused on SurrealDB 3.x** and the field **dropped on 2.x** and the embedded engine, silently: declare the field on
the model. The README's 2.x/3.x table has the row ("A write naming a field a `SCHEMAFULL` table does not define").

In [6]:
from surrealdb import ServerError

if major >= 3:
    try:
        await stored.merge(nickname="Countess", mood="happy")  # mood is not a field of Person
    except ServerError as error:
        print("SurrealDB 3.x refuses the write:", type(error).__name__, "-", error)
else:
    await stored.merge(nickname="Countess", mood="happy")
    print("SurrealDB 2.x accepts the write and drops mood")
print("person:ada:", await demo.query("SELECT nickname, mood FROM ONLY person:ada;", {}))

SurrealDB 3.x refuses the write: InternalError - Found field 'mood', but no such field exists for table 'person'
person:ada: {'mood': None, 'nickname': None}


## 7. A table tuned by hand

`note` is defined by hand first: `SCHEMAFULL`, a comment, a change feed, permissions, an `ASSERT` and a `COMMENT` on
`email`, `DEFAULT time::now() READONLY` on `created`. `Note` declares none of it, and `define_table()` keeps it: a
clause the model leaves out is carried from the stored definition, while the `TYPE` always comes from the model
(`email: str | None` is `option<string>`). Applied by hand, the bare `table_ddl()` would reset those clauses to
SurrealDB's defaults: `SCHEMALESS`, no change feed, `PERMISSIONS NONE`.

`created` is in `server_fields` (v0.18.0): `save()` leaves it out, and the database's `DEFAULT` fills it.

In [7]:
await apply_ddl(
    demo,
    "DEFINE TABLE note SCHEMAFULL COMMENT 'keep' CHANGEFEED 1h PERMISSIONS FOR select WHERE owner = $auth.id;"
    "DEFINE FIELD email ON note TYPE string ASSERT $value != '' COMMENT 'mail';"
    "DEFINE FIELD created ON note TYPE option<datetime> DEFAULT time::now() READONLY;",
)
print(await Note.define_table())
print()
note = await Note(id="n1", body="Buy milk").save()
print("created by the database:", note.created is not None, "| done:", note.done)

DEFINE TABLE OVERWRITE `note` TYPE NORMAL SCHEMAFULL COMMENT 'keep' CHANGEFEED 1h PERMISSIONS FOR select WHERE owner = $auth.id, FOR create, update, delete NONE;
DEFINE FIELD OVERWRITE `body` ON `note` TYPE string;
DEFINE FIELD OVERWRITE `email` ON `note` TYPE option<string> ASSERT $value != '' COMMENT 'mail';
DEFINE FIELD OVERWRITE `created` ON `note` TYPE option<datetime> DEFAULT time::now() READONLY;
DEFINE FIELD OVERWRITE `done` ON `note` TYPE bool DEFAULT false;

created by the database: True | done: False


What the model declares wins. A model on the same table declaring `comment=` replaces the stored comment and keeps the
rest; `schema_mode=SchemaMode.SCHEMALESS` would turn the table back, and `permissions={}` writes `PERMISSIONS NONE`. A
key left out never removes a stored clause: remove a change feed or a comment with raw SurrealQL.

In [8]:
class DocumentedNote(Note):
    model_config = SurrealConfigDict(table_name="note", comment="Notes, documented by the model")


print(await DocumentedNote.define_table())
print()
print(await table_definition("note"))

DEFINE TABLE OVERWRITE `note` TYPE NORMAL SCHEMAFULL COMMENT 'Notes, documented by the model' CHANGEFEED 1h PERMISSIONS FOR select WHERE owner = $auth.id, FOR create, update, delete NONE;
DEFINE FIELD OVERWRITE `body` ON `note` TYPE string;
DEFINE FIELD OVERWRITE `email` ON `note` TYPE option<string> ASSERT $value != '' COMMENT 'mail';
DEFINE FIELD OVERWRITE `created` ON `note` TYPE option<datetime> DEFAULT time::now() READONLY;
DEFINE FIELD OVERWRITE `done` ON `note` TYPE bool DEFAULT false;

DEFINE TABLE note TYPE NORMAL SCHEMAFULL COMMENT 'Notes, documented by the model' CHANGEFEED 1h PERMISSIONS FOR select WHERE owner = $auth.id, FOR create, update, delete NONE


## 8. What depends on the server line

On a `SCHEMAFULL` table, SurrealDB 2.x and the embedded engine empty the objects and arrays of an `any` column and of
an untyped list, silently (`{x: {y: 1}}` is stored `{}`), unless the column is `FLEXIBLE`; 3.x keeps them, and refuses
`FLEXIBLE` there. SurrealDB 3.x also refuses an object item of a required untyped list unless its items are defined.
2.x and the embedded engine also drop the value of a `FLEXIBLE` column under a list of objects — the `dict` of a
sub-model in a `list[M]` — from every item, unless the list is `FLEXIBLE` too. So `define_table()` writes the item
definition (`items.* TYPE any`) on every line, and reads the line (`client.version()`, once, only for a `SCHEMAFULL`
table with a column that holds `any` or such a list) to write `FLEXIBLE` on those columns and lists before 3.x; the
items' typed fields are still checked. `table_ddl()`, which needs no connection, renders the 3.x form.

`Loose` is a `SCHEMAFULL` model with an `Any`, an untyped `list` and a list of sub-models holding a `dict`: its values
round-trip on both lines.

In [9]:
class Line(BaseModel):
    sku: str
    meta: dict[str, Any] = Field(default_factory=dict)


class Loose(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="loose", connection="tdl_demo", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    anything: Any = None
    items: list = Field(default_factory=list)
    lines: list[Line] = Field(default_factory=list)


print(f"Server: SurrealDB {major}.x")
sql = await Loose.define_table()
print(sql)
print()
print("the statements of table_ddl():", sql == "\n".join(Loose.table_ddl()))
if major >= 3:
    print("  SurrealDB 3.x: define_table() applies table_ddl()'s statements")
else:
    print("  SurrealDB 2.x: define_table() adds FLEXIBLE to the columns that hold any and to the list of sub-models")

loose = Loose(id="l1", anything={"x": {"y": 1}}, items=[{"k": 1}, 2], lines=[Line(sku="a", meta={"k": {"j": 1}})])
await loose.save()
print("read back equal:", (await Loose.objects().get("l1")).model_dump() == loose.model_dump())

Server: SurrealDB 3.x
DEFINE TABLE OVERWRITE `loose` TYPE NORMAL SCHEMAFULL;
DEFINE FIELD OVERWRITE `anything` ON `loose` TYPE any;
DEFINE FIELD OVERWRITE `items` ON `loose` TYPE array;
DEFINE FIELD OVERWRITE `items`.* ON `loose` TYPE any;
DEFINE FIELD OVERWRITE `lines` ON `loose` TYPE array<object>;
DEFINE FIELD OVERWRITE `lines`.*.`sku` ON `loose` TYPE string;
DEFINE FIELD OVERWRITE `lines`.*.`meta` ON `loose` TYPE object FLEXIBLE;

the statements of table_ddl(): True
  SurrealDB 3.x: define_table() applies table_ddl()'s statements
read back equal: True


On a `SCHEMALESS` table no line empties an object or an array, so no column is written `FLEXIBLE` there (3.x refuses
it on a `SCHEMALESS` table): `Log`'s statements are the same on both lines.

In [10]:
print(await Log.define_table())
entry = Log(id="e1", payload={"user": {"id": 7}}, detail=[{"step": 1}, "done"])
await entry.save()
print("read back equal:", (await Log.objects().get("e1")).model_dump() == entry.model_dump())

DEFINE TABLE OVERWRITE `log` TYPE ANY SCHEMALESS;
DEFINE FIELD OVERWRITE `payload` ON `log` TYPE object;
DEFINE FIELD OVERWRITE `detail` ON `log` TYPE any;
read back equal: True


## 9. Edges and views

An edge model's `define_table()` applies the relation statement `define_relation()` (v0.33.0) builds — the declared
clauses after the `TYPE RELATION` clause — then the edge's data fields, never `in` and `out`, which SurrealDB defines
from `IN` and `OUT`. `define_relation()` defines no field, so for an edge model that declares
`SchemaMode.SCHEMAFULL` and has data fields it raises before any request: every `relate()` with data would be refused
(3.x) or lose its data (2.x).

In [11]:
from surreal_orm_lite import SurrealDbError

print(await Likes.define_table())
print()
edge = await stored.relate(Likes, note, stars=5)
print("edge:", edge.in_, "->", edge.out, "| stars:", edge.stars)
print()
try:
    await Likes.define_relation()
except SurrealDbError as error:
    print(error)

DEFINE TABLE OVERWRITE `likes` TYPE RELATION IN `person` OUT `note` SCHEMAFULL PERMISSIONS FOR select FULL, FOR create, update, delete NONE;
DEFINE FIELD OVERWRITE `stars` ON `likes` TYPE int DEFAULT 1;

edge: person:ada -> note:n1 | stars: 5

Likes.define_relation(): Likes declares SchemaMode.SCHEMAFULL and has data fields ('stars'); define_relation() defines no field, and a SCHEMAFULL table refuses (SurrealDB 3.x) or drops (2.x and the embedded engine) the fields it does not define. Call Likes.define_table(), which applies the same relation statement and defines them. Nothing was defined.


A view model's `define_table()` applies `define_view()` (v0.32.0), its guards included, and returns its statement. No
field is defined on a view: SurrealDB computes its records. The declared comment and permissions are written after
the `AS (…)` clause.

In [12]:
print(await NoteStats.define_table())
view = (await DatabaseIntrospector("tdl_demo", tables=["note_stats"]).introspect()).tables["note_stats"]
print("comment:", view.comment, "| permissions:", dict(view.permissions))
print("notes counted:", [row.n for row in await NoteStats.objects().all()])

DEFINE TABLE OVERWRITE `note_stats` AS (SELECT count() AS n FROM note GROUP ALL) COMMENT 'Counts' PERMISSIONS FOR select FULL, FOR create, update, delete NONE;
comment: Counts | permissions: {'create': 'NONE', 'delete': 'NONE', 'select': 'FULL', 'update': 'NONE'}
notes counted: [1]


## 10. User models

`define_access()` (v0.17.0) reads the same schema mode as `define_table()`. A model with `table_type=TableType.USER`
(the full ORM's key for an auth table) or `schema_mode=SchemaMode.SCHEMAFULL` gets from `define_access()` a
`SCHEMAFULL` table with one `DEFINE FIELD` per field and the access's permissions, then the access: without the field
definitions, a `SCHEMAFULL` table would refuse (3.x) or empty (2.x) the record SIGNUP creates. Signup and signin work
on both lines.

In [13]:
from surreal_orm_lite import AuthenticatedUserMixin


class Member(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="member", connection="tdl_demo", table_type=TableType.USER)

    id: str | None = None
    email: str
    password: str
    name: str = ""


for statement in await Member.define_access():
    print(statement)
print()
created = await Member.signup(email="ada@example.com", password="correct-horse", name="Ada")
signed_in = await Member.signin(email="ada@example.com", password="correct-horse")
print("signed up:", created.user.email, "| signed in:", signed_in.user.email, "| name:", signed_in.user.name)

DEFINE TABLE OVERWRITE `member` TYPE NORMAL SCHEMAFULL PERMISSIONS FOR select, update WHERE id = $auth.id, FOR create, delete NONE;
DEFINE FIELD OVERWRITE `email` ON `member` TYPE string;
DEFINE FIELD OVERWRITE `password` ON `member` TYPE string;
DEFINE FIELD OVERWRITE `name` ON `member` TYPE string DEFAULT '';
DEFINE ACCESS OVERWRITE member_access ON DATABASE TYPE RECORD SIGNUP ( CREATE member SET email = $email, password = crypto::argon2::generate($password), name = $name ) SIGNIN ( SELECT * FROM member WHERE email = $email AND crypto::argon2::compare(password, $password) ) DURATION FOR TOKEN 15m, FOR SESSION 12h;



signed up: ada@example.com | signed in: ada@example.com | name: Ada


`define_table()` afterwards writes the same table and fields, and keeps the access's permissions, which `Member` does
not declare: nothing changes. Any call order converges — `define_table()` first, then `define_access()`, gives the
same table.

In [14]:
async def member_table() -> tuple[str, dict]:
    """The table's definition and its fields, as INFO FOR DB and INFO FOR TABLE read them back."""
    return await table_definition("member"), (await demo.query("INFO FOR TABLE member;", {}))["fields"]


before = await member_table()
print(await Member.define_table())
print()
print("nothing changed:", await member_table() == before)

DEFINE TABLE OVERWRITE `member` TYPE NORMAL SCHEMAFULL PERMISSIONS FOR select, update WHERE id = $auth.id, FOR create, delete NONE;
DEFINE FIELD OVERWRITE `email` ON `member` TYPE string;
DEFINE FIELD OVERWRITE `password` ON `member` TYPE string;
DEFINE FIELD OVERWRITE `name` ON `member` TYPE string DEFAULT '';

nothing changed: True


## 11. Transactions

`define_table(tx=tx)` reads the definitions first, on committed state outside the block, then adds its statements to
the block: applied at once in an interactive transaction (WebSocket, SurrealDB 3.x), at commit in a buffered one
(SurrealDB 2.x, HTTP, the embedded engine). A block that raises applies nothing. A definition of the same table or
field queued earlier in the block is not seen, and is replaced at commit: define it before the block.

In [15]:
class Draft(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="draft", connection="tdl_demo", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    title: str


class Rollback(Exception):
    """Raised to roll the block back."""


try:
    async with SurrealDBConnectionManager.transaction("tdl_demo") as tx:
        print(await Draft.define_table(tx=tx))
        raise Rollback
except Rollback:
    print("the block raised")
print("draft defined after the block:", "draft" in (await demo.query("INFO FOR DB;", {}))["tables"])

DEFINE TABLE OVERWRITE `draft` TYPE NORMAL SCHEMAFULL;
DEFINE FIELD OVERWRITE `title` ON `draft` TYPE string;
the block raised
draft defined after the block: False


## 12. The guards

`define_table()` refuses, before any DDL, the cases where its statements would not do what the model asks: a node
model over a materialized view (an ordinary table statement would drop its query) or over a relation table (it would
drop the relation constraint and refuse every edge), and a table or field definition it cannot read back (it could not
keep the clauses the model does not declare). Each raises `SurrealDbError`, naming the table, and nothing is defined.

A computed field's expression (v0.16.0) is checked before any request too: a SurrealQL comment marker would swallow
the clauses written after it. `table_ddl()` raises `ValueError`.

In [16]:
from surreal_orm_lite import Computed, computed


class Liking(BaseSurrealModel):  # a node model given the relation table of Likes by mistake
    model_config = SurrealConfigDict(table_name="likes", connection="tdl_demo")

    id: RecordID | None = None
    stars: int = 1


try:
    await Liking.define_table()
except SurrealDbError as error:
    print(error)
print()


class Cart(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="cart", connection="tdl_demo")

    id: str | None = None
    total: Computed[int] = computed("1 + 1 -- note")


try:
    Cart.table_ddl()
except ValueError as error:
    print(error)

Liking.define_table(): the table 'likes' is a relation table (TYPE RELATION) and Liking is not an edge model; defining it as an ordinary table would drop its relation constraint and refuse every edge. Derive Liking from BaseSurrealEdge, or give it another table_name. Nothing was defined.

Cart: the computed field 'total' is not SurrealQL the ORM can embed: it holds a SurrealQL comment marker (-- , #, // or /* */).


## 13. Round trip

`generate_models_from_db()` (v0.30.0) generates models carrying the keys — `table_type`, `schema_mode`,
`changefeed`, `permissions`, `comment` — so a generated model's `define_table()` writes the same table clauses back:
type, schema mode, comment, change feed (as SurrealDB renders it: `1w`) and permissions. The file is written to a
temporary directory, whose path is not printed.

In [17]:
import pathlib
import tempfile

from surreal_orm_lite import generate_models_from_db

with tempfile.TemporaryDirectory() as directory:
    path = pathlib.Path(directory) / "models.py"
    print(await generate_models_from_db(path, "tdl_demo", tables=["person", "note", "log", "likes"]))

"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from datetime import datetime
from typing import Any, Literal

from pydantic import BaseModel
from surreal_orm_lite import (
    BaseSurrealEdge,
    BaseSurrealModel,
    SchemaMode,
    SurrealConfigDict,
    TableType,
)


class Likes(BaseSurrealEdge):
    model_config = SurrealConfigDict(
        table_name="likes",
        relation_in="person",
        relation_out="note",
        schema_mode=SchemaMode.SCHEMAFULL,
        permissions={"select": "FULL"},
        connection="tdl_demo",
    )

    stars: int = 1


class Log(BaseSurrealModel):
    # SCHEMALESS: fields not dec

## 14. Cleanup

Remove the `examples_table_ddl` database, unregister `tdl_demo` — `remove_connection()` closes its client — and close
every connection with `close_all_connections()`.

In [18]:
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_table_ddl;")
await SurrealDBConnectionManager.remove_connection("tdl_demo")
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

Connections:

 ['default']
Done
